In [ ]:
##################################################
#
# NEURAL NETWORK (1 HIDDEN LAYER, REGRESSION)
#
#################################################
#
# Neural network is nested compositon of functions - f(x) = f_L(...f_2(f_1(x)))
# Where weights are obtained via backpropogation
# Backprop is reverse-mode auto differentiation - uses the directed acyclic graph structure to avoid recomputation
# Gradient of loss wrt weights of any layer would depend on all layers between that layer and output
#
# The decision boundary here isnt a hyperplane - its piecewise linear function (ReLU) or smooth nonlinear manifold (sigmoid)
# Capacity isnt controlled by parameter count now, its controlled by how many pieces the function can break into
# NN can overfit even with few parameters, and why margin theory alone doesn't explain generalization.
#
# NN can have depth (many layers) or width (many units per layer)
# Depth allows hierarchial feature compostion
# Width at layer 1: learns h different linear combinations of x.
# Depth (layer 2 uses layer 1's output): learns combinations of combinations.
# Ex: Layer 1 learns edges , 2 learns corners, 3 learns shapes
# A 3 layer network can represent some functions that a 2 layer network needs exponentially many units to represent
#
# NN uses smooth activations to get gradients to flow backwards
# sigmoid / tanh have gradients that vanish when inputs are too large (above a saturation limit)
# If weights push activations into saturation zone (some |z| >= c) gradients die -> no updation occurs
# ReLU fixes this for positive inputs, for negative it gives dead neurons
# These conditioning issues, to solve need to look at stuff gradients depend on
# Gradient magnitude depends on product of weight matrices - ∂L/∂W1 involves (W2^T)(W3^T)...(W_L^T) times something - singular values matter
# If largest singular value (spectral norm) > 1 grad grows, < 1 grad shrinks.
# We can control how these are initialized : He initialization: W ~ N(0, 2/fan_in) for ReLU, Xavier: W ~ N(0, 1/fan_in) for sigmoid/tanh.


# Hypothesis class:
# f(x; W1, b1, W2, b2) = W2 * σ(W1*x + b1) + b2
#
# f is a non-linear differentiable function
# MSE loss with L2 regulaisation (same strength) used here
# 
# ReLU - Rectified linear unit:
# σ(z) = max(0, z)
# σ'(z) = 1 if z > 0 else 0
# No saturation for z > 0
# Gradient is 1 - not shrinking
# Computationally cheap
# Dead neurons if z <= 0 always

In [2]:
import numpy as np

In [ ]:
class NeuralNetwork:
    def __init__(self, input_dim, hidden_dim, lr=0.01, reg_str=0.0):
        """
        Input:  X ∈ R^(n × d)      — n samples, d features
        W1:     W1 ∈ R^(h × d)     — first layer weights
        b1:     b1 ∈ R^h           — first layer bias
        z1:     z1 ∈ R^(n × h)     — pre-activation hidden
        a1:     a1 ∈ R^(n × h)     — post-activation hidden
        W2:     W2 ∈ R^(h × 1)     — output weights , column vector 
        b2:     b2 ∈ R             — output bias , scalar
        y_hat:  y_hat ∈ R^(n × 1)  — predictions

        """
        self.lr = lr 
        self.reg_str = reg_str
        self.W1 = np.random.randn(hidden_dim, input_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, 1) * np.sqrt(2.0 / hidden_dim)
        self.b2 = 0.0

    def relu(self,z):
        return np.maximum(0,z)
    def relu_deriv(self,z):
        return (z>0).astype(float)
    
    def forward(self,X):
        """
        Forward pass
        z1 = X @ W1.T + b1           (n × d) @ (d × h) = (n × h)
        a1 = σ(z1)                   element-wise activation
        z2 = a1 @ W2 + b2            (n × h) @ (h × 1) = (n × 1)
        y_hat = z2                   no output activation for regression

        """
        self.X = X
        self.z1 = X @ self.W1.T + self.b1 
        self.a1 = self.relu(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        return self.z2
    
    def mse_reg_loss(self,y_pred,y):
        """ 
        MSE Loss 
        L = (1/n) * sum((y_pred - y)²)
        
        L2 regularisation
        L_total = L_data + (λ/2) * (||W1||² + ||W2||²)
        ∂L/∂W1 ← ∂L/∂W1 + λ * W1
        ∂L/∂W2 ← ∂L/∂W2 + λ * W2

        """

        mse_loss  = np.mean((y_pred-y)**2)
        reg_loss  = (self.reg_str/2.0) * (np.sum(self.W1 **2) + np.sum(self.W2**2))
        return mse_loss + reg_loss
    
    def backward(self,y_pred,y):
        
        n = len(y)

        # δ2 = (2/n) * (y_hat - y)          # (n × 1)
        dL_dz2 = (2.0/n) * (y_pred - y)

        # ∂L/∂W2 = a1.T @ δ2 + (reg_2 * W2) # (h × n) @ (n × 1) = (h × 1)
        # ∂L/∂b2 = sum(δ2)                  # scalar
        dL_dW2= self.a1.T @ dL_dz2 + self.reg_str * self.W2
        dL_db2 = np.sum(dL_dz2)

        # ∂L/∂a1 = δ2 @ W2.T                # (n × 1) @ (1 × h) = (n × h)
        # δ1 = ∂L/∂a1 ⊙ σ'(z1)             # (n × h) element-wise
        dL_dz1 = (dL_dz2 @ self.W2.T) * self.relu_deriv(self.z1)

        # ∂L/∂W1 = δ1.T @ X (reg_1 * W1)    # (h × n) @ (n × d) = (h × d)
        # ∂L/∂b1 = sum(δ1, axis=0)          # (h,)
        dL_dW1 = dL_dz1.T @ self.X + self.reg_str * self.W1
        dL_db1 = np.sum(dL_dz1,axis=0)

        self.W1 -= self.lr * dL_dW1
        self.b1 -= self.lr * dL_db1
        self.W2 -= self.lr * dL_dW2
        self.b2 -= self.lr * dL_db2

    def fit(self, X, y, epochs=1000):
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        
        losses = []
        for epoch in range(epochs):
            y_hat = self.forward(X)
            loss = self.mse_reg_loss(y_hat, y)
            losses.append(loss)
            self.backward(y_hat, y)
            
        return losses
    
    def predict(self, X):
        return self.forward(X)
# Training Example
np.random.seed(42)
X = np.random.uniform(-3, 3, (200, 1))
y = X**2 + np.random.normal(0, 0.5, (200, 1))

X_train, y_train = X[:150], y[:150]
X_test, y_test = X[150:], y[150:]

nn = NeuralNetwork(input_dim=1, hidden_dim=10, lr=0.01, reg_str=0.01)
losses = nn.fit(X_train, y_train, epochs=2000)

y_pred = nn.predict(X_test)
test_loss = np.mean((y_pred - y_test)**2)
print(f"\nTest MSE: {test_loss:.6f}")


Test MSE: 0.267596
